making sure that our env had been setup correct

In [29]:
import sys
import torch
import ultralytics
import cv2

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)

Python: c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\.venv\Scripts\python.exe
PyTorch: 2.15.0+cpu
Ultralytics: 8.4.213
OpenCV: 5.0.0


# Fire and Smoke Detection Using YOLO

## 1. Dataset Preparation

In this section, we connect to Roboflow and download the Fire and Smoke Detection dataset.

**Dataset details:**
- Workspace: gfo-fc1de
- Project: fire-and-smoke-detector-yykya
- Version: 14
- Format: YOLO

In [30]:
import os
from pathlib import Path

from dotenv import load_dotenv
from roboflow import Roboflow

# Identify project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load environment variables from .env file
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    raise ValueError("ROBOFLOW_API_KEY was not found in the .env file.")

print("Environment configuration loaded successfully.")



Environment configuration loaded successfully.


In [31]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env", override=True)

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

assert ROBOFLOW_API_KEY, "Roboflow API key is missing."

print("API key loaded successfully.")

API key loaded successfully.


In [32]:
# Initialize Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Select workspace and project
WORKSPACE_NAME = "gfo-fc1de"
PROJECT_NAME = "fire-and-smoke-detector-yykya"
VERSION_NUMBER = 14 


workspace = rf.workspace(WORKSPACE_NAME)
project = workspace.project(PROJECT_NAME)
version = project.version(VERSION_NUMBER)


# Define dataset location
DATASET_DIR = PROJECT_ROOT / "data" / "raw" / "fire_smoke"

# Download dataset in Ultralytics YOLO format
dataset = version.download(
    "yolo26",
    location=str(DATASET_DIR)
)


print("Dataset downloaded successfully!")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Project: {PROJECT_NAME}")
print(f"Version: {VERSION_NUMBER}")
print(f"Dataset location: {dataset.location}")




loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded successfully!
Workspace: gfo-fc1de
Project: fire-and-smoke-detector-yykya
Version: 14
Dataset location: c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\data\raw\fire_smoke


In [33]:
# Check dataset directory
assert DATASET_DIR.exists(), "Dataset directory was not created."

# Check data.yaml
DATA_YAML = DATASET_DIR / "data.yaml"

assert DATA_YAML.exists(), "data.yaml was not found."

print("Dataset structure:")
for path in sorted(DATASET_DIR.iterdir()):
    print(path.name)

print("\ndata.yaml content:")
print(DATA_YAML.read_text())

Dataset structure:
data.yaml
README.dataset.txt
README.roboflow.txt
test
train
valid

data.yaml content:
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['fire', 'smoke']

roboflow:
  workspace: gfo-fc1de
  project: fire-and-smoke-detector-yykya
  version: 14
  license: CC BY 4.0
  url: https://universe.roboflow.com/gfo-fc1de/fire-and-smoke-detector-yykya/dataset/14


In [58]:
import yaml

# Load dataset configuration
with open(DATA_YAML, "r") as file:
    data_config = yaml.safe_load(file)

# Check dataset paths
for split in ["train", "val", "test"]:

    parent_file = DATA_YAML.parent.name

    split_path = (DATA_YAML.parent / parent_file / data_config[split]).resolve()

    print(f"{split}: {split_path}")
    print(f"Exists: {split_path.exists()}")
    print("-" * 50)


train: C:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\data\raw\fire_smoke\train\images
Exists: True
--------------------------------------------------
val: C:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\data\raw\fire_smoke\valid\images
Exists: True
--------------------------------------------------
test: C:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\data\raw\fire_smoke\test\images
Exists: True
--------------------------------------------------


## 2. Dataset Inspection

In this section, we inspect the downloaded dataset to understand its
organization, image distribution, and annotation quality before training.

In [60]:
from collections import Counter

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

splits = ["test", "train", "val"]

dataset_summary = {}

for split in splits:

    images_dir = (DATA_YAML.parent / DATA_YAML.parent.name / data_config[split]).resolve()
    labels_dir = images_dir.parent / "labels"

    images = [
        file for file in images_dir.iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []

    dataset_summary[split] = {
        "images": len(images),
        "labels": len(labels)
    }

    print(f"\n{split.upper()}")
    print(f"Images: {len(images)}")
    print(f"Label files: {len(labels)}")


TEST
Images: 639
Label files: 639

TRAIN
Images: 3940
Label files: 3940

VAL
Images: 1017
Label files: 1017


In [61]:
class_names = data_config["names"]

for split in splits:

    images_dir = (DATA_YAML.parent /  DATA_YAML.parent.name / data_config[split]).resolve()
    labels_dir = images_dir.parent.parent / "labels"

    class_counts = Counter()

    for label_file in labels_dir.glob("*.txt"):

        with open(label_file, "r") as file:

            for line in file:

                values = line.strip().split()

                if values:
                    class_id = int(values[0])
                    class_counts[class_id] += 1

    print(f"\n{split.upper()}")

    for class_id, class_name in enumerate(class_names):
        print(f"{class_name}: {class_counts[class_id]}")


TEST
fire: 0
smoke: 0

TRAIN
fire: 0
smoke: 0

VAL
fire: 0
smoke: 0


In [62]:
for split in splits:

    images_dir = (DATA_YAML.parent / DATA_YAML.parent.name /data_config[split]).resolve()
    labels_dir = images_dir.parent.parent / "labels"

    images = [
        file for file in images_dir.iterdir()
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    missing_labels = []
    empty_labels = []

    for image_path in images:

        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            missing_labels.append(image_path.name)

        elif label_path.stat().st_size == 0:
            empty_labels.append(image_path.name)

    print(f"\n{split.upper()}")
    print(f"Missing annotations: {len(missing_labels)}")
    print(f"Empty annotations: {len(empty_labels)}")


TEST
Missing annotations: 639
Empty annotations: 0

TRAIN
Missing annotations: 3940
Empty annotations: 0

VAL
Missing annotations: 1017
Empty annotations: 0


In [66]:
from pathlib import Path

DATASET_DIR = Path(PROJECT_ROOT / "data" / "raw" / "fire_smoke")

for split in ["train", "valid", "test"]:
    split_dir = DATASET_DIR / split

    print(f"\n{split.upper()}")

    for item in split_dir.iterdir():
        if item.is_dir():
            print(f"Directory: {item.name}")
            print(f"Files: {len(list(item.iterdir()))}")
        else:
            print(f"File: {item.name}")


TRAIN
Directory: images
Files: 3940
Directory: labels
Files: 3940

VALID
Directory: images
Files: 1017
Directory: labels
Files: 1017

TEST
Directory: images
Files: 639
Directory: labels
Files: 639


In [69]:
YOLO_DATA_CONFIG = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": data_config["nc"],
    "names": data_config["names"]
}

YOLO_DATA_CONFIG

{'path': 'c:\\Users\\mahmoud local\\Desktop\\Studying\\Projects\\fire_smoke_detection\\data\\raw\\fire_smoke',
 'train': 'train/images',
 'val': 'valid/images',
 'test': 'test/images',
 'nc': 2,
 'names': ['fire', 'smoke']}

In [70]:
import yaml

YOLO_DATA_YAML = DATASET_DIR / "yolo_data.yaml"

with open(YOLO_DATA_YAML, "w") as file:
    yaml.safe_dump(
        YOLO_DATA_CONFIG,
        file,
        sort_keys=False
    )

print(f"Configuration saved: {YOLO_DATA_YAML}")

Configuration saved: c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\data\raw\fire_smoke\yolo_data.yaml


In [ ]:
from ultralytics import YOLO



MODEL_PATH = PROJECT_ROOT / "models" / "fire_smoke_yolo26s.pt"

model = YOLO(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


In [81]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

VIDEO_DIR = PROJECT_ROOT / "outputs" / "videos"
PREDICTIONS_DIR = PROJECT_ROOT / "outputs" / "predictions"

print("Project root:", PROJECT_ROOT)
print("Video directory exists:", VIDEO_DIR.exists())
print("Predictions directory:", PREDICTIONS_DIR)

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_FILES = list(VIDEO_DIR.glob("*"))

print("Available videos:")

for video in VIDEO_FILES:
    if video.suffix.lower() in [".mp4", ".avi", ".mov", ".mkv"]:
        print(video.name)

Project root: c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection
Video directory exists: True
Predictions directory: c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\outputs\predictions
Available videos:
224207_medium.mp4
44187-438212438_medium.mp4


In [84]:
VIDEO_PATH = VIDEO_DIR / "44187-438212438_medium.mp4"

print("Selected video:", VIDEO_PATH)
print("Exists:", VIDEO_PATH.exists())

Selected video: c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\outputs\videos\44187-438212438_medium.mp4
Exists: True


In [86]:
results = model.predict(
    source=str(VIDEO_PATH),
    save=True,
    project=str(PREDICTIONS_DIR),
    name="fire_smoke_detection",
    exist_ok=True,
    conf=0.1,
    imgsz=640,
    show=False
)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/495) c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\outputs\videos\44187-438212438_medium.mp4: 384x640 1 smoke, 81.0ms
video 1/1 (frame 2/495) c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\outputs\videos\44187-438212438_medium.mp4: 384x640 1 smoke, 82.9ms
video 1/1 (frame 3/495) c:\Users\mahmoud local\Desktop\Studying\Projects\fire_smoke_detection\outputs\videos\44187-438212438_medium.mp4: 384